# Project 24: Mobile App Usage & User Engagement Analysis
**Google Colab-ready | Python | Pandas | NumPy | Matplotlib**

Dataset: `24_mobile_app_usage.csv`

This notebook analyzes mobile app sessions, feature usage, device types, user age groups, ratings, usage timing, outliers, and engagement. It follows the 15 tasks listed in Project 24.

## 1. Import libraries and load dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# In Colab, upload the CSV if it is not already available.
try:
    from google.colab import files
    import os
    if not os.path.exists("24_mobile_app_usage.csv"):
        uploaded = files.upload()
except ImportError:
    pass

df = pd.read_csv("24_mobile_app_usage.csv")
print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())


## 2. Inspect dimensions, columns, and data types

In [ ]:
print("Rows and columns:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nData types:")
print(df.dtypes)
print("\\nDataset information:")
df.info()


## 3. Missing values and duplicate Session_ID records

In [ ]:
print("Missing values by column:")
display(df.isnull().sum())

print("Duplicate Session_ID records:", df["Session_ID"].duplicated().sum())


## 4. Clean data and prepare date/time fields
Duplicate Session_ID rows are removed. Missing duration is filled with the median. Missing rating remains missing because it should not be guessed.

In [ ]:
df = df.drop_duplicates(subset="Session_ID").copy()

df["Session_Date"] = pd.to_datetime(df["Session_Date"], errors="coerce")
df["Session_Duration_Mins"] = pd.to_numeric(df["Session_Duration_Mins"], errors="coerce")
df["User_Rating"] = pd.to_numeric(df["User_Rating"], errors="coerce")

median_duration = df["Session_Duration_Mins"].median()
df["Session_Duration_Mins"] = df["Session_Duration_Mins"].fillna(median_duration)

df["Month"] = df["Session_Date"].dt.month_name()
df["Weekday"] = df["Session_Date"].dt.day_name()
df["Hour"] = df["Session_Date"].dt.hour
df["Year"] = df["Session_Date"].dt.year

print("Cleaned shape:", df.shape)
display(df.head())


## 5. Total sessions, unique users, and average session duration

In [ ]:
total_sessions = len(df)
unique_users = df["User_ID"].nunique()
average_duration = df["Session_Duration_Mins"].mean()

print("Total sessions:", total_sessions)
print("Unique users:", unique_users)
print("Average session duration (minutes):", round(average_duration, 2))


## 6. Top 10 most frequently used app features

In [ ]:
feature_popularity = df["Feature_Used"].value_counts().head(10)
display(feature_popularity.to_frame("Session_Count"))


## 7. Average session duration by device type

In [ ]:
device_comparison = df.groupby("Device_Type")["Session_Duration_Mins"].mean().sort_values(ascending=False)
display(device_comparison.to_frame("Average_Duration_Mins"))


## 8. User age groups ranked by session count

In [ ]:
age_group_sessions = df["Age_Group"].value_counts().sort_values(ascending=False)
display(age_group_sessions.to_frame("Session_Count"))


## 9. Average user ratings across app features

In [ ]:
feature_ratings = df.groupby("Feature_Used")["User_Rating"].mean().sort_values(ascending=False)
display(feature_ratings.to_frame("Average_Rating"))


## 10. Peak usage hours and busiest weekdays

In [ ]:
hourly_usage = df["Hour"].value_counts().sort_index()
weekday_usage = df["Weekday"].value_counts()

print("Peak hour:", hourly_usage.idxmax(), "with", hourly_usage.max(), "sessions")
print("Busiest weekday:", weekday_usage.idxmax(), "with", weekday_usage.max(), "sessions")

display(hourly_usage.to_frame("Sessions"))
display(weekday_usage.to_frame("Sessions"))


## 11. List, tuple, and set

In [ ]:
most_used_features = feature_popularity.index.tolist()
fixed_device_categories = ("Android", "iOS", "Other")
unique_users_set = set(df["User_ID"].dropna())

print("Top features list:", most_used_features)
print("Device categories tuple:", fixed_device_categories)
print("Number of unique users in set:", len(unique_users_set))


## 12. Function: average session duration for a selected feature

In [ ]:
def average_duration_for_feature(feature_name):
    feature_rows = df[df["Feature_Used"] == feature_name]
    if feature_rows.empty:
        return "Feature not found"
    return round(feature_rows["Session_Duration_Mins"].mean(), 2)

# Example:
print("Average duration for a selected feature:", average_duration_for_feature("Chat"))


## 13. NumPy statistics for session duration

In [ ]:
durations = df["Session_Duration_Mins"].dropna().to_numpy()

print("Mean:", round(np.mean(durations), 2))
print("Median:", round(np.median(durations), 2))
print("Standard deviation:", round(np.std(durations), 2))


## 14. IQR outliers and engagement labels
Engagement labels used here: **Low** = under 10 minutes; **Medium** = 10–30 minutes; **High** = over 30 minutes.

In [ ]:
q1 = np.percentile(durations, 25)
q3 = np.percentile(durations, 75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df[(df["Session_Duration_Mins"] < lower_bound) |
               (df["Session_Duration_Mins"] > upper_bound)]

print("Q1:", q1, "Q3:", q3, "IQR:", iqr)
print("Outlier boundaries:", lower_bound, "to", upper_bound)
print("Outlier session count:", len(outliers))
display(outliers)

def engagement_label(minutes):
    if minutes < 10:
        return "Low"
    elif minutes <= 30:
        return "Medium"
    else:
        return "High"

df["Engagement"] = df["Session_Duration_Mins"].apply(engagement_label)
display(df["Engagement"].value_counts().to_frame("Session_Count"))


## 15. AppUser class (OOP)

In [ ]:
class AppUser:
    def __init__(self, user_id, device_type):
        self.user_id = user_id
        self.device_type = device_type

    def show_details(self):
        print("User ID:", self.user_id)
        print("Device Type:", self.device_type)

sample_user = AppUser("USR001", "Android")
sample_user.show_details()


## 16. Charts and graphs

In [ ]:
# Feature popularity chart
feature_popularity.plot(kind="bar", figsize=(10, 5), title="Top App Features by Session Count")
plt.xlabel("Feature")
plt.ylabel("Number of Sessions")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Average session duration by device
device_comparison.plot(kind="bar", figsize=(7, 5), title="Average Session Duration by Device")
plt.xlabel("Device Type")
plt.ylabel("Minutes")
plt.tight_layout()
plt.show()

# Sessions by hour
hourly_usage.plot(kind="line", marker="o", figsize=(9, 5), title="App Sessions by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Sessions")
plt.xticks(range(0, 24))
plt.grid(True)
plt.tight_layout()
plt.show()


## 17. Five recommendations to improve app engagement

In [ ]:
recommendations = [
    "Improve the most frequently used features and make them easy to access.",
    "Investigate features with lower average ratings and improve their user experience.",
    "Optimize app performance for device types with shorter average session duration.",
    "Use peak-hour patterns to plan notifications and server capacity.",
    "Add simple onboarding tips to help users discover useful features."
]
for number, recommendation in enumerate(recommendations, start=1):
    print(f"{number}. {recommendation}")


## 18. Save cleaned dataset

In [ ]:
df.to_csv("24_mobile_app_usage_cleaned.csv", index=False)
print("Cleaned CSV saved.")

# In Colab, uncomment the next lines to download the cleaned CSV:
# from google.colab import files
# files.download("24_mobile_app_usage_cleaned.csv")


## Conclusion
This project analyzes app engagement through session duration, feature popularity, device type, age group, ratings, and usage timing. The results can help prioritize feature improvements, app performance, and user experience changes.